### Here we Train MiniBERT using Masked Language Modeling (MLM)

In [2]:
import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt

MAX_ITERS = 5000

EVAL_INTERVAL = 250

EVAL_ITERS = 100

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 0.01

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [3]:
# Importing MiniBERT from notebook 9
import importlib

# 1. Use importlib to dynamically load the notebook 9 module
notebook09_module = importlib.import_module("ipynb.fs.full.09_MiniBERT-Model")

# 2. Extract the class from the loaded module
MiniBERT = notebook09_module.MiniBERT


torch.Size([2, 10, 768])
torch.Size([2, 128, 256])
7,332,744
torch.Size([4, 128, 5000])


In [4]:
VOCAB_SIZE = 5000

EMBED_DIM = 256

MAX_LEN = 128

NUM_HEADS = 8

NUM_LAYERS = 6

DROPOUT = 0.1

model = MiniBERT(

    vocab_size=VOCAB_SIZE,

    embed_dim=EMBED_DIM,

    max_len=MAX_LEN,

    num_heads=NUM_HEADS,

    num_layers=NUM_LAYERS

)

In [5]:
model = model.to(device)

In [6]:
optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)

In [7]:
# Loss Function
def compute_loss(

    logits,

    labels
):

    B,T,C = logits.shape

    logits = logits.view(
        B*T,
        C
    )

    labels = labels.view(
        B*T
    )

    loss = F.cross_entropy(

        logits,

        labels,

        ignore_index=-100

    )

    return loss

In [8]:
# Validation Function
@torch.no_grad()
def estimate_loss():

    model.eval()

    out = {}

    for split in [

        "train",

        "val"

    ]:

        losses = torch.zeros(
            EVAL_ITERS
        )

        for k in range(
            EVAL_ITERS
        ):

            xb,labels = get_batch(
                split
            )

            xb = xb.to(device)

            labels = labels.to(device)

            logits = model(
                xb
            )

            loss = compute_loss(
                logits,
                labels
            )

            losses[k] = (
                loss.item()
            )

        out[split] = (
            losses.mean()
        )

    model.train()

    return out

In [9]:
train_losses = []

val_losses = []

In [12]:
# Importing GetBatch from notebook 10
import importlib

# 1. Use importlib to dynamically load the notebook 10 module
notebook10_module = importlib.import_module("ipynb.fs.full.10_BERT-DataPipelineMLM")

# 2. Extract the class from the loaded module
get_batch = notebook10_module.get_batch


cuda
5000
False
tensor([ 407,  765,   12, 1975,  116, 2271,  424, 1915,    8,  395,   81,  365,
          10,  882,   12, 2030,    8,  365,   10,  407])
tensor([ 407,  765,   12, 1975,    4,    4,    4, 1915,    8,  395,   81,  365,
          10,  882,   12,    4,    8,  365,   10,  407])
tensor([-100, -100, -100, -100,  116, 2271,  424, -100, -100, -100, -100, -100,
        -100, -100, -100, 2030, -100, -100, -100, -100])
torch.Size([16, 128])
torch.Size([16, 128])
tensor(329)
cuda:0


In [13]:
# Training Loop
for step in range(
    MAX_ITERS
):

    if (
        step %
        EVAL_INTERVAL
        == 0
    ):

        losses = estimate_loss()

        train_losses.append(
            losses["train"]
        )

        val_losses.append(
            losses["val"]
        )

        print(

            f"Step {step}"

            f" | Train Loss "

            f"{losses['train']:.4f}"

            f" | Val Loss "

            f"{losses['val']:.4f}"

        )

    xb,labels = get_batch(
        "train"
    )

    xb = xb.to(device)

    labels = labels.to(device)

    logits = model(
        xb
    )

    loss = compute_loss(
        logits,
        labels
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    optimizer.step()

Step 0 | Train Loss 8.6752 | Val Loss 8.6830
Step 250 | Train Loss 6.4758 | Val Loss 6.5176
Step 500 | Train Loss 6.4633 | Val Loss 6.5547
Step 750 | Train Loss 6.4679 | Val Loss 6.5879
Step 1000 | Train Loss 6.4478 | Val Loss 6.5442
Step 1250 | Train Loss 6.4443 | Val Loss 6.5548
Step 1500 | Train Loss 6.4825 | Val Loss 6.5458
Step 1750 | Train Loss 6.4712 | Val Loss 6.5523
Step 2000 | Train Loss 6.4585 | Val Loss 6.5427
Step 2250 | Train Loss 6.4673 | Val Loss 6.5388
Step 2500 | Train Loss 6.4720 | Val Loss 6.5567
Step 2750 | Train Loss 6.4712 | Val Loss 6.5408
Step 3000 | Train Loss 6.4612 | Val Loss 6.5273
Step 3250 | Train Loss 6.4655 | Val Loss 6.5394
Step 3500 | Train Loss 6.4550 | Val Loss 6.5857
Step 3750 | Train Loss 6.4415 | Val Loss 6.5612
Step 4000 | Train Loss 6.4635 | Val Loss 6.5586
Step 4250 | Train Loss 6.4623 | Val Loss 6.5799
Step 4500 | Train Loss 6.4687 | Val Loss 6.5604
Step 4750 | Train Loss 6.4440 | Val Loss 6.5630


In [ ]:
torch.save(

    model.state_dict(),

    "mini_bert.pt"

)

print(
    "MiniBERT Saved"
)

In [ ]:
plt.figure(
    figsize=(10,5)
)

plt.plot(
    train_losses,
    label="Train"
)

plt.plot(
    val_losses,
    label="Validation"
)

plt.xlabel(
    "Evaluation Step"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "MiniBERT MLM Training"
)

plt.legend()

plt.show()

In [ ]:
checkpoint = {

    "model_state_dict":
    model.state_dict(),

    "optimizer_state_dict":
    optimizer.state_dict()

}

In [ ]:
torch.save(

    checkpoint,

    "mini_bert_checkpoint.pt"

)

In [ ]:
# MLM Prediction Demo
xb,labels = get_batch(
    "val"
)
xb = xb.to(device)

with torch.no_grad():

    logits = model(
        xb
    )

preds = torch.argmax(
    logits,
    dim=-1
)

print(
    preds[0][:20]
)